# TB-Trust — 01: Data, manifest, and degradation ablation (Phase 2)

Builds the manifest from the raw images, checks the per-clinic class balance (which decides what can serve as a held-out fold), and runs the degradation ablation.

In [ ]:
# --- configuration ---------------------------------------------------------
# Defaults are the Kaggle paths. Every path is read from the environment first,
# so the same notebook runs unmodified on Kaggle, locally, or in CI -- which is
# also what lets these notebooks be executed as a test rather than only read.
import os

REPO = os.environ.get("TBTRUST_REPO", "/kaggle/working/tb-trust")
DATA = os.environ.get("TBTRUST_DATA", "/kaggle/input/tuberculosis-tb-chest-xray-dataset")
WORK = os.environ.get("TBTRUST_WORK", "/kaggle/working")
REPO_URL = os.environ.get("TBTRUST_REPO_URL", "https://github.com/AIscend-Research/tb-trust.git")

MANIFEST = f"{WORK}/manifest.csv"
OUT = f"{WORK}/outputs"
os.makedirs(WORK, exist_ok=True)
print("REPO:", REPO, "\nDATA:", DATA, "\nWORK:", WORK)

In [ ]:
# Enter the repo and make it importable. The install is skipped when the package
# already resolves, so re-running a notebook is cheap.
import importlib.util
import os
import subprocess
import sys

os.chdir(REPO)
sys.path.insert(0, os.path.join(REPO, "src"))
if importlib.util.find_spec("tbtrust") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
    importlib.invalidate_caches()
print("tbtrust ready from", REPO)

## 1. Build the manifest

Handles the aggregated Kaggle layout (`Normal/`, `Tuberculosis/` folders) and the raw NLM filename convention (`MCUCXR_*_0/1`, `CHNCXR_*_0/1`).

In [ ]:
import subprocess
import sys

r = subprocess.run(
    [sys.executable, "scripts/build_manifest.py", "--raw", DATA, "--out", MANIFEST],
    capture_output=True, text=True,
)
print(r.stdout or r.stderr)
assert r.returncode == 0, "build_manifest failed"

## 2. Class balance decides which folds are usable

Only clinics holding **both** classes can be held out: on a single-class test set sensitivity or specificity is undefined, and the clinic label becomes a near-perfect proxy for the diagnosis. NIAID and Belarus are TB-only, RSNA is normal-only — so Montgomery and Shenzhen are the two-class holdouts.

In [ ]:
from tbtrust.data import manifest as M

df = M.load(MANIFEST)
print(M.class_balance_report(df))
print("\ntotal images:", len(df), "| columns:", list(df.columns))

## 3. Leave-one-clinic-out folds

In [ ]:
from tbtrust.data.splits import all_loco_folds, leave_one_clinic_out, summarize_split

folds = all_loco_folds(df, two_class_only=True)
print("usable two-class LOCO folds:", folds)
assert folds, "no two-class clinic in the manifest -- check the class balance above"

split = leave_one_clinic_out(df, holdout_clinic=folds[0], seed=0)
print(f"\nsplit sizes with {folds[0]} held out:")
print(summarize_split(split))

## 4. What the smartphone degradation actually does

Seven ops with continuous severity. `total_severity` averages over the possible op slots, so an image with more simultaneous artifacts scores higher — it is the signal-to-noise proxy the weak uncertainty label is built from.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

from tbtrust.data.degradation import SmartphoneDegradation

src = np.asarray(Image.open(df["path"].iloc[0]).convert("L").resize((256, 256)))
severities = [0.0, 0.25, 0.5, 0.75, 1.0]

fig, axes = plt.subplots(1, len(severities), figsize=(3 * len(severities), 3.4))
for ax, s in zip(axes, severities, strict=True):
    img, rec = SmartphoneDegradation(severity=s, seed=0)(src)
    ax.imshow(img, cmap="gray")
    ax.set_title(f"severity {s}\ntotal={rec.total_severity:.2f}", fontsize=9)
    ax.axis("off")
fig.tight_layout()
plt.show()

## 5. Degradation-strategy ablation

Scores the physics pipeline against the learned generator. With no real phone recaptures attached it can only compare the strategies to each other — pass `--real-dir` once you have a recapture set (see `data/real_recapture/README.md`).

In [ ]:
import subprocess
import sys

r = subprocess.run(
    [sys.executable, "scripts/ablate_degradation.py",
     "--source", DATA, "--n", "40", "--severity", "0.7", "--skip-learned",
     "--out", f"{WORK}/degradation_ablation.json"],
    capture_output=True, text=True,
)
print((r.stdout or r.stderr)[-2500:])
assert r.returncode == 0, "ablate_degradation failed"

Drop `--skip-learned` to include the adversarially-trained degrader (needs torch and is much slower), and add `--real-dir <path>` to score both against real phone recaptures — that comparison is the one that makes the ablation meaningful.

Next: **02_train_models.ipynb**.